# ERM* + DPE: neural aggregation

Enable Internet and select **GPU T4 ×2**. Run seeds 0 and 1 on separate GPUs.
The repository contains all implementation and configuration files; this notebook
only prepares the checkout and launches the experiment. Publish the experiment
files to the selected repository branch before running it on Kaggle.


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/chautanphat/waterbirds-shortcut-mitigation.git'
REPO_BRANCH = 'experiment/dpe-ermstar-neural'  # Branch containing scripts/run_dpe_ermstar.py.
REPO = Path('/kaggle/working/waterbirds-shortcut-mitigation')
WORK = Path('/kaggle/working/dpe_ermstar')
SEEDS = [0, 1]
DATA_DIR = None
RESUME_FROM = None  # Previous full jobs/ directory, if resuming.
TIME_BUDGET_HOURS = 10


In [ ]:
import os
import subprocess
import sys

if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch',
                    REPO_URL, str(REPO)], check=True)
if not (REPO / 'scripts/run_dpe_ermstar.py').is_file():
    raise RuntimeError('This checkout lacks the experiment. Publish the repository changes or select their branch.')
if 'src' in sys.modules and not Path(sys.modules['src'].__file__).resolve().is_relative_to(REPO):
    raise RuntimeError('Restart the kernel before using this repository.')
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import torch
if torch.cuda.device_count() < 2:
    raise RuntimeError('Select GPU T4 ×2 in Kaggle Settings.')
print([torch.cuda.get_device_name(i) for i in range(2)])


In [ ]:
from src.methods.dpe_ermstar.run import prepare_and_launch

results = prepare_and_launch(
    WORK, seeds=SEEDS, data_dir=DATA_DIR, resume_from=RESUME_FROM,
    config_path=REPO / 'configs/dpe_ermstar/study.json',
    timeout_hours=TIME_BUDGET_HOURS,
)


In [ ]:
import numpy as np

for name in results[0]['results']:
    values = [r['results'][name]['metrics']['wga'] for r in results]
    print(name, 'mean WGA (%)', 100 * np.mean(values),
          'per seed (%)', [100 * v for v in values])
print('Result archives:', list((WORK / 'jobs').glob('*-results.zip')))


The configuration follows the upstream **400-epoch** ERM* launcher. Both arms
share a backbone: full-validation annotated DPE uniform, and separate 720/479
head/combiner pools with matched uniform and neural aggregation. Report each
configuration independently; this is exploratory work.

For recovery, retain the full jobs directory, repository checkout, and notebook.
Backbones resume at epoch boundaries; DPE resumes at head boundaries. The 400-epoch
runs may require another Kaggle session. See `src/methods/dpe_ermstar/README.md`.
